In [1]:
import os
import json
import subprocess
import tempfile
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, PrecisionRecallDisplay

In [ ]:
# ==============================================================================
# 1. SETUP & PATH CONFIGURATION
# ==============================================================================
RUN_ACQUIRE_PATH = Path("/home/azureuser/EBRD_Tillage/EBRD_tillage_monitoring-main/src/acquire/run_acquire.py")  
TEMP_DIR = Path("/home/azureuser/EBRD_Tillage/data/temp")
TEMP_DIR.mkdir(parents=True, exist_ok=True)


INPUT_DATA_PATH= Path("/home/azureuser/EBRD_Tillage/data/input")  
def resolve_fields_path(path):
    p = Path(path)
    if not p.is_dir():
        return p
    for pattern in ("*.shp", "*.gpkg", "*.geojson"):
        matches = sorted(p.glob(pattern))
        if matches:
            return matches[0]
    raise FileNotFoundError(f"No vector file found in: {p}")

shapefile_path = resolve_fields_path(INPUT_DATA_PATH)
print(f"Reading shapefile from: {shapefile_path}")
gdf = gpd.read_file(shapefile_path)
# Quick check
print(f"Successfully loaded {len(gdf)} rows.")
print(gdf.head())

In [2]:
gdf.columns

NameError: name 'gdf' is not defined

In [ ]:


# ==============================================================================
# 2. ITERATE OVER TILLAGE EVENTS & EXECUTE ACQUISITION PIPELINE
# ==============================================================================
event_ndvi_results = []

for idx, row in gdf.iterrows():
    event_id = str(row["fid"])
    till_type = row["till_type"]
    
    # Apply 10-day buffer to event dates
    buf_start = (row["start_date"] - pd.Timedelta(days=10)).strftime("%Y-%m-%d")
    buf_end = (row["end_date"] + pd.Timedelta(days=10)).strftime("%Y-%m-%d")
    
    # Save isolated event geometry to temp file
    event_gdf = gpd.GeoDataFrame([row], crs=gdf.crs)
    event_vector_path = TEMP_DIR / f"{event_id}.geojson"
    event_gdf.to_file(event_vector_path, driver="GeoJSON")
    
    # Build dynamic JSON config
    event_output_dir = TEMP_DIR / f"output_{event_id}"
    config_dict = {
        "fields_path": str(event_vector_path),
        "field_id_col": "fid",
        "output_dir": str(event_output_dir),
        "start_date": buf_start,
        "end_date": buf_end,
        "max_cloud_cover": 80,
        "resolution": 10,
        "assets": ["B02", "B03", "B04", "B05", "B06", "B07", "B08", "B8A", "B11", "B12", "SCL"],
        "max_workers": 2,
        "skip_existing": True
    }
    
    config_path = TEMP_DIR / f"config_{event_id}.json"
    with open(config_path, "w", encoding="utf-8") as f:
        json.dump(config_dict, f, indent=2)
        
    print(f"\n--- Processing Event {idx+1}/{len(gdf)}: {event_id} ({till_type}) | Window: {buf_start} to {buf_end} ---")
    
    # Call run_acquire.py via subprocess CLI
    cmd = [
        "python", str(RUN_ACQUIRE_PATH),
        "--source", "s2",
        "--config", str(config_path),
        "--steps", "inventory", "download"
    ]
    
    try:
        subprocess.run(cmd, check=True)
    except subprocess.CalledProcessError as e:
        print(f"Acquisition failed for event {event_id}: {e}")
        event_ndvi_results.append(np.nan)
        continue

    # ==========================================================================
    # 3. CALCULATE NDVI & TWO-STAGE AGGREGATION (TEMPORAL FIRST, THEN SPATIAL)
    # ==========================================================================
    parquet_files = list((event_output_dir / "scene_samples").glob("**/*.parquet"))
    
    if not parquet_files:
        print(f"No scene samples returned for event {event_id}.")
        event_ndvi_results.append(np.nan)
        continue

    # Load and combine extracted pixel data
    df_pixels = pd.concat([pd.read_parquet(f) for f in parquet_files], ignore_index=True)
    
    # Filter valid non-cloudy pixels
    df_valid = df_pixels[df_pixels["valid_px"] == True].copy()
    if df_valid.empty:
        print(f"Warning: All pixels invalid/cloudy. Falling back to unmasked pixels for {event_id}.")
        df_valid = df_pixels.copy()
        
    # Calculate NDVI: (NIR - Red) / (NIR + Red) -> (B8 - B4) / (B8 + B4)
    df_valid["ndvi"] = (df_valid["B8"] - df_valid["B4"]) / (df_valid["B8"] + df_valid["B4"] + 1e-6)
    
    # Stage 1: Temporal Aggregation First (min NDVI over time window for each pixel location)
    pixel_min_ndvi = df_valid.groupby(["pixel_col", "pixel_row"])["ndvi"].min()
    
    # Stage 2: Spatial Aggregation (spatial mean across field polygon)
    event_summary_ndvi = pixel_min_ndvi.mean()
    
    event_ndvi_results.append(event_summary_ndvi)
    print(f"Resulting Event NDVI (Temporal Min -> Spatial Mean): {event_summary_ndvi:.4f}")

# Attach summary NDVI values back to primary GeoDataFrame
gdf["summary_ndvi"] = event_ndvi_results

# Filter out failed/missing rows for threshold analysis
analysis_df = gdf.dropna(subset=["summary_ndvi"]).copy()

# ==============================================================================
# 4. CANDIDATE THRESHOLDING OPTIMIZATION
# ==============================================================================
# Encode target: Conventional = 0, Conservation = 1
# (Conservation tillage typically retains higher post-tillage NDVI residue)
analysis_df["binary_target"] = (analysis_df["till_type"].str.lower() == "conservation").astype(int)

y_true = analysis_df["binary_target"].values
scores = analysis_df["summary_ndvi"].values

# Evaluate candidate thresholds across the full score spectrum
candidate_thresholds = np.linspace(scores.min(), scores.max(), 200)

best_f1 = -1.0
best_acc = -1.0
best_threshold = candidate_thresholds[0]

f1_scores = []
acc_scores = []

for t in candidate_thresholds:
    y_pred = (scores >= t).astype(int)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    acc = accuracy_score(y_true, y_pred)
    
    f1_scores.append(f1)
    acc_scores.append(acc)
    
    # Optimize primarily for F1 score (and accuracy as secondary metric)
    if f1 > best_f1 or (f1 == best_f1 and acc > best_acc):
        best_f1 = f1
        best_acc = acc
        best_threshold = t

print("\n================ THRESHOLD OPTIMIZATION RESULTS ================")
print(f"Optimal NDVI Threshold: {best_threshold:.4f}")
print(f"Maximized F1-Score:     {best_f1:.4f}")
print(f"Overall Accuracy:       {best_acc:.4f}")
print("================================================================\n")

# ==============================================================================
# 5. VISUALIZATION OF DISTRIBUTIONS AND PERFORMANCE CURVES
# ==============================================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Class Distributions & Threshold Split
conv_scores = analysis_df[analysis_df["binary_target"] == 0]["summary_ndvi"]
cons_scores = analysis_df[analysis_df["binary_target"] == 1]["summary_ndvi"]

ax1.hist(conv_scores, bins=15, alpha=0.6, color="orangered", label="Conventional")
ax1.hist(cons_scores, bins=15, alpha=0.6, color="seagreen", label="Conservation")
ax1.axvline(best_threshold, color="black", linestyle="--", linewidth=2, 
            label=f"Optimal Threshold ({best_threshold:.3f})")
ax1.set_xlabel("Summary NDVI (Temporal Min → Spatial Mean)")
ax1.set_ylabel("Count")
ax1.set_title("NDVI Distribution by Tillage Type")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Metrics vs Candidate Thresholds
ax2.plot(candidate_thresholds, f1_scores, color="purple", label="F1-Score", linewidth=2)
ax2.plot(candidate_thresholds, acc_scores, color="dodgerblue", label="Accuracy", linewidth=2)
ax2.axvline(best_threshold, color="black", linestyle="--", linewidth=1.5, label="Optimal Cutoff")
ax2.set_xlabel("Candidate Threshold")
ax2.set_ylabel("Score")
ax2.set_title("Model Optimization vs NDVI Threshold")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()